# CHIMESim

Interactive collective movement and reproducible datasets, using shared scenarios. Install from this checkout with `python -m pip install -e ".[masks,notebook]"`. Select a Python kernel with Tkinter to open either GUI.

In [ ]:
import chimesim

## Interactive exploration

Start/Pause in Controls, change sliders, draw walls and explore individual variation. The last scenario and window arrangement are saved automatically. Close the visualiser before running the next cell.

In [ ]:
chimesim.interactive()

## An independent scenario

`source="defaults"` makes this example independent of previous interactive sessions. Scenarios contain scientific settings; run duration and repeats are provided separately.

In [ ]:
scenario = chimesim.settings(source="defaults", model="zonal",
    environment={"width": 1200, "height": 800, "bounded": True},
    movement={"desired_speed": 3, "fov_deg": 270, "visual_range": 200},
    agents={"A1": {"n": 80, "speed_scale": 0.8, "color": "#56b1fc"},
            "A2": {"n": 40, "speed_scale": 1.3, "color": "#f4a030"}})

In [ ]:
chimesim.interactive(settings=scenario, seed=42)

## Individual differences

Types can carry distributions over several traits. Sampled individual values stay fixed through time. Social speed and speed noise are separate options for temporal changes. Zone radii must remain ordered for each individual.

In [ ]:
heterogeneous = chimesim.settings(source=scenario, agents={
    "A1": {"n": 100, "social_speed": 0.3, "distributions": {
        "speed_scale": {"dist": "uniform", "low": 0.7, "high": 1.3},
        "r_a": {"dist": "normal", "mean": 75, "std": 10,
                "clip": [40, 110], "truncate": True}}}})

In [ ]:
chimesim.interactive(settings=heterogeneous, seed=42)

## Relationships between types

Types are general groups, with optional directed relationships. A2 can pursue A1 while A1 flees A2. Pursuit never removes agents: capture is a separate opt-in choice, available in each type's Relationships controls. Both steering responses respect visual range, FOV, walls and the maximum turn. Types without a directed relationship school together.

This example leaves capture off. To enable it, set A2's `captures` to `["A1"]`; `movement.capture_distance` controls the contact distance. Older saved `hunts` settings now load as pursuit without automatic removal.

In [ ]:
relationships = chimesim.settings(source="defaults",
    movement={"visual_range": 200, "capture_distance": 10},
    agents={"A1": {"n": 100, "flees": ["A2"], "flee_dist": 150},
            "A2": {"n": 5, "speed_scale": 1.4, "attraction": 200,
                   "pursues": ["A1"], "captures": []}})

In [ ]:
chimesim.interactive(settings=relationships, seed=42)

## Generate datasets with the setup GUI

This separate window configures dataset generation without opening the interactive visualiser. Set the duration, burn-in, repeats and output folder, then click **Simulate**. The Conditions tab offers parameter dropdowns: every combination is run, with repeats for each combination. Three speeds and three attraction radii therefore give nine conditions.

**Stop simulation** preserves data already written; **Exit** closes the window. Once you exit, `gui_run` contains the most recent run's folder and status (or `None` if nothing ran). Each run gets its own folder.

In [ ]:
gui_run = chimesim.simulate(gui=True, settings=heterogeneous,
    n_steps=1000, burn_in=200, n_reps=5, performance=True)

### Read and plot saved GUI results

The same loading and plotting functions accept the returned run information or a saved run-folder path. This plot uses the summaries already saved during simulation; it does not need raw trajectories or additional processing.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from chimesim import analysis
from chimesim.plotting import plot

if gui_run is not None:
    print(gui_run["output_dir"])
    gui_tables = analysis.load(gui_run)
    display(gui_tables["summary"].head())
    plot(gui_run, "polarisation", category="rep", time_bin=25)
# Later: analysis.load(Path.home() / "CHIMESim-results" / "run-YYYYMMDD-HHMMSS-...")

## Generate datasets directly in Python

This example runs **five repeats**, each with **200 burn-in steps followed by 1,000 recorded steps**. The seed makes the starting conditions reproducible. `performance=True` prints timing and workload rates and includes computer/software details in `results["performance"]`.

With no explicit settings, `simulate()` uses the last interactive scenario; the explicit scenario below is independent of that session. The returned dictionary contains summaries, repeat means, static individual traits, trajectories and the settings needed to reproduce the run.

In [ ]:
results = chimesim.simulate(settings=heterogeneous, n_steps=1000, burn_in=200,
    n_reps=5, seed=42, save_trajectories=True, performance=True)

In [ ]:
print("Result entries:", list(results))
display(pd.DataFrame({"dataset": name, "rows": len(results[name])}
    for name in ["summary", "repeats", "individuals", "trajectories"]))

In [ ]:
summary = pd.DataFrame(results["summary"])
summary.head()

In [ ]:
trajectories = pd.DataFrame(results["trajectories"])
trajectories.head()

## Starter processing and plots

Requires the notebook or analysis extra. Processing uses the largest physical group(s) in each frame. Front ranks run from rear 0 to front 1; each individual gets its average over valid samples. Under identical, well-mixed behaviour, individual averages should approach 0.5; they are not forced to do so. Weakly polarised groups have no front score.

Compare the five repeats with separate colours. The wide polarisation plot uses thin lines and **25-step bin means** to make the temporal pattern easier to see; the raw data are unchanged. Set `time_bin=None` to show every recorded frame.

In [ ]:
import matplotlib.pyplot as plt

processed = analysis.process(results)
fig, ax = plt.subplots(figsize=(12, 3.5))
plot(processed, "polarisation", kind="time", category="rep", time_bin=25, ax=ax)
plt.show()

The next plots compare **repeats**, since this example has only one condition and one agent type. `unit="frame"` shows the temporal distribution of the selected group's mean centroid distance or mean speed within each repeat. These are descriptive distributions of correlated frames, not independent samples. For comparisons between experimental conditions, use `category="condition"` with the default `unit="repeat"` to compare repeat means.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
plot(processed, "centroid_distance", kind="box", category="rep", unit="frame", ax=axes[0])
plot(processed, "speed", kind="density", category="rep", unit="frame", ax=axes[1])
fig.tight_layout()
plt.show()

For an established schooling run with valid front scores, use `plot(processed, "front_rank", kind="scatter", x="speed_scale", table="individuals")` to inspect traits versus average spatial positions. Use `table="positions"` for individual time histories. Boxes/densities average within repeats by default; choose `unit="individual"` to plot individual mean positions, or `unit="frame"` explicitly for frame distributions. Tables are pandas dataframes and can be exported to CSV for R.

## Conditions (parameter grid)

Three A1 speed multipliers and three A1 attraction radii make **nine combinations**, with **two repeats per combination**: 18 simulations in total. Each repeat records 100 steps after 50 burn-in steps. The table below lists the parameter values, number of repeats and actual rows stored for each combination. Summary rows describe model steps; trajectory rows describe individual agents at those steps.

In [ ]:
grid = chimesim.simulate(settings=scenario,
    conditions={"agents.A1.speed_scale": [0.5, 1.0, 1.5], "agents.A1.attraction": [90, 120, 150]},
    n_steps=100, burn_in=50, n_reps=2, seed=42, save_trajectories=True,
    performance=True)

In [ ]:
condition_table = pd.DataFrame({"condition": item["condition"], **item["values"]}
    for item in grid["conditions"])
condition_table = condition_table.merge(
    pd.DataFrame(grid["summary"]).groupby("condition", as_index=False).agg(
        repeats=("rep", "nunique"), summary_rows=("frame", "size")), on="condition")
condition_table = condition_table.merge(
    pd.DataFrame(grid["trajectories"]).groupby("condition").size().rename("trajectory_rows"),
    on="condition")
condition_table.rename(columns={"agents.A1.speed_scale": "A1 speed multiplier",
    "agents.A1.attraction": "A1 attraction radius"})

In [ ]:
plot(grid, "polarisation", kind="box", category=["condition", "agent_type"])

## Save and reproduce

Store the scenario together with run controls to retain the actual seed, including when the original seed was generated automatically. For long runs, pass `output_dir="results"` to stream CSV datasets and configuration into a new run folder.

In [ ]:
import json
from pathlib import Path
configuration = {"scenario": results["settings"], "run": results["run"]}
Path("experiment.json").write_text(json.dumps(configuration, indent=2))
# Reproduce later: chimesim.simulate(settings="experiment.json")

## Stream a batch to files for R or later analysis

`output_dir` is a parent folder, not a CSV filename. Each call creates a new run subfolder, shared by its conditions and repeats. Relative paths use the current working directory; this example uses an absolute home-directory path. Omitting `output_dir` keeps the Python workflow above. Set `save_trajectories=False` for summary outputs without raw tracks.

In [ ]:
run = chimesim.simulate(settings=scenario,
    conditions={"agents.A1.n": [50, 100, 200, 400]},
    n_steps=1000, burn_in=200, n_reps=3, seed=42,
    save_trajectories=True, output_dir=Path.home() / "CHIMESim-results")
print(run["output_dir"])

The folder contains `trajectories.csv`, `individuals.csv`, `summary.csv`, `repeats.csv` and configuration/status files. Join tracks and traits by `set` and `id`; each set is one condition/repeat. Trajectories contain `id`, `x`, `y`, numeric model-step time `t`, frame, velocities and heading in radians; they are suitable raw inputs for a tracking-analysis workflow. In periodic arenas x/y wrap, so downstream distance/velocity calculations must handle the arena topology. See the guide for R import and swaRmverse formatting notes.

## Explore milling and schooling

The [Couzin et al. (2002) model](https://jmvidal.cse.sc.edu/library/couzin02a.pdf) is three-dimensional. This is a **2D exploratory starting point**, not a verified milling preset. A compact swarm with individuals turning both ways is different from coherent milling: check for low polarisation **and sustained high milling**, using the Information window.

Start with a single type and no trait variation, social speed or speed noise. Keep repulsion omnidirectional (360°), social FOV at 270°, and lock the arena size when resizing. Let the group settle, then change orientation gradually while retaining the current group: try radii 15–35 around the starting value of 20. Explore maximum turns of 4–8° per step separately. Small changes can select a different collective state, and restarting removes the group's history. These ranges are suggestions to explore, not a guarantee of milling.

In [ ]:
milling = chimesim.settings(source="defaults", model="zonal",
    environment={"width": 1600, "height": 1000, "bounded": False, "spawn_radius": 70},
    movement={"desired_speed": 3, "fov_deg": 270, "repulsion_fov_deg": 360, "visual_range": 140},
    agents={"A1": {"n": 100, "repulsion": 10, "orientation": 20, "attraction": 140,
                   "max_turn_deg": 4, "noise": 0.05, "heading_noise_memory": 0,
                   "social_speed": 0, "speed_noise": 0}})

In [ ]:
chimesim.interactive(settings=milling, seed=42)

Here the zone values are **radii**, not zone widths. At speed 3 and a maximum turn of 4° per step, an agent's minimum steady-turn radius is about 43 model units; a very tight aggregation cannot turn as a coherent ring at that scale. Attraction, repulsion, alignment and initial conditions all matter. The original paper also reports hysteresis, so independent condition-grid runs do not reproduce its parameter-continuation experiment. A reproducible milling demonstration still needs validation in this implementation; the distinct Calovi/Gautrais angular-velocity model remains a future addition.

See [the user guide](docs/usage.md) for parameter definitions, environment drawing, metrics, recording and advanced perception development.